In [1]:
import os
import pandas as pd
import numpy as np

# 원본 데이터 로드
df = pd.read_csv('UniversalBank.csv')
print(f"데이터 로드 완료: {df.shape[0]}행 × {df.shape[1]}열")

데이터 로드 완료: 5000행 × 10열


In [2]:
# 전처리 전 상태 확인
print(f"원본 데이터: {df.shape[0]}행 × {df.shape[1]}열")
print(f"결측값 총 수: {df.isnull().sum().sum()}")
print(f"중복 행 수: {df.duplicated().sum()}")

원본 데이터: 5000행 × 10열
결측값 총 수: 0
중복 행 수: 0


In [3]:
# 원본 보존 후 전처리 시작
df_clean = df.copy()

# ID 컬럼 제거 — 인덱스 변수로 분석에 불필요
df_clean = df_clean.drop(columns=['ID'])
print(f"ID 컬럼 제거 완료 → {df_clean.shape[1]}열")

ID 컬럼 제거 완료 → 9열


In [4]:
# Experience 음수값 처리
# EDA에서 Age ≈ Experience + 상수 관계 확인 → Age 기반으로 대체
# 음수 경력은 데이터 입력 오류로 판단

neg_exp_mask = df_clean['Experience'] < 0
neg_count = neg_exp_mask.sum()
print(f"Experience 음수값: {neg_count}건")

if neg_count > 0:
    # Age - Experience의 중앙값(offset)으로 추정하여 대체
    offset = (df_clean.loc[~neg_exp_mask, 'Age'] - df_clean.loc[~neg_exp_mask, 'Experience']).median()
    df_clean.loc[neg_exp_mask, 'Experience'] = (df_clean.loc[neg_exp_mask, 'Age'] - offset).clip(lower=0).round().astype(int)
    print(f"Experience 음수값 → Age 기반 대체 완료 (offset={offset:.1f})")
    print(f"대체 후 min: {df_clean['Experience'].min()}")

Experience 음수값: 52건
Experience 음수값 → Age 기반 대체 완료 (offset=25.0)
대체 후 min: 0


In [5]:
# Mortgage 피처 분리
# 중위수=0, 약 70%가 0 → 보유 여부(이진)와 금액(연속)으로 분리

df_clean['HasMortgage'] = (df_clean['Mortgage'] > 0).astype(int)
print(f"HasMortgage 분포:\n{df_clean['HasMortgage'].value_counts()}")
print(f"\nMortgage 금액 (보유자만): {df_clean[df_clean['HasMortgage']==1]['Mortgage'].describe()}")

HasMortgage 분포:
HasMortgage
0    3462
1    1538
Name: count, dtype: int64

Mortgage 금액 (보유자만): count    1538.000000
mean      183.676203
std       101.361226
min        75.000000
25%       109.000000
50%       153.000000
75%       227.000000
max       635.000000
Name: Mortgage, dtype: float64


In [6]:
# 범주형 인코딩 — CDAccount, PLoan: Yes/No → 1/0
df_clean['CDAccount'] = df_clean['CDAccount'].map({'Yes': 1, 'No': 0})
df_clean['PLoan'] = df_clean['PLoan'].map({'Yes': 1, 'No': 0})
print("CDAccount, PLoan 인코딩 완료 (Yes=1, No=0)")
print(df_clean[['CDAccount', 'PLoan']].value_counts())

CDAccount, PLoan 인코딩 완료 (Yes=1, No=0)
CDAccount  PLoan
0          0        4358
           1         340
1          0         162
           1         140
Name: count, dtype: int64


In [7]:
# 전처리 완료 요약 + 저장
print(f"원본 데이터:  {df.shape[0]}행 × {df.shape[1]}열")
print(f"전처리 후:   {df_clean.shape[0]}행 × {df_clean.shape[1]}열")
print(f"결측값: {df_clean.isnull().sum().sum()}")
print(f"\n최종 dtypes:")
print(df_clean.dtypes)

# 전처리 데이터 저장
os.makedirs('cleaned', exist_ok=True)
df_clean.to_csv('cleaned/UniversalBank_cleaned.csv', index=False)
print(f"\n저장 완료: cleaned/UniversalBank_cleaned.csv")

원본 데이터:  5000행 × 10열
전처리 후:   5000행 × 10열
결측값: 0

최종 dtypes:
Age              int64
Experience       int64
Income           int64
Family           int64
CCAvg          float64
Education        int64
CDAccount        int64
Mortgage         int64
PLoan            int64
HasMortgage      int64
dtype: object

저장 완료: cleaned/UniversalBank_cleaned.csv
